In [31]:
import numpy as np
import pandas as pd

from pathlib import Path

from itertools import product
from sklearn.metrics import f1_score


from scipy.special import softmax

In [32]:
ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
OOF_DIR = ROOT_DIR / "oof"

SUBMISSION_DIR = ROOT_DIR / "submission"
SUBMISSION_DIR.mkdir(exist_ok=True)

In [33]:
conv_logits_test = np.load(OOF_DIR / "convnext_test_logits.npy")
swin_logits_test = np.load(OOF_DIR / "swin_test_logits.npy")
siglip_logits_test = np.load(OOF_DIR / "siglip_test_logits.npy")

In [34]:
p_conv_test = softmax(conv_logits_test, axis=1)
p_swin_test = softmax(swin_logits_test, axis=1)
p_siglip_test = softmax(siglip_logits_test, axis=1)

In [35]:
manual = pd.read_csv("manual_label.csv")
manual.head()

,id,label
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0


In [36]:
y_true = manual["label"].values

In [37]:
best_score = -1
best_weight = None

results = []

step = 0.05

weights = np.arange(0, 1 + step, step)

for w1 in weights:

    for w2 in weights:

        w3 = 1 - w1 - w2

        if w3 < 0:
            continue

        probs = (
            w1 * p_conv_test +
            w2 * p_swin_test +
            w3 * p_siglip_test
        )

        pred = probs.argmax(axis=1)

        score = f1_score(
            y_true,
            pred,
            average="macro"
        )

        results.append(
            [w1, w2, w3, score]
        )

        if score > best_score:

            best_score = score

            best_weight = (w1, w2, w3)

print("=" * 60)

print("Best Score :", best_score)

print()

print("ConvNeXt :", best_weight[0])
print("Swin      :", best_weight[1])
print("SigLIP    :", best_weight[2])

Best Score : 0.9595199044250805

ConvNeXt : 0.55
Swin      : 0.35000000000000003
SigLIP    : 0.09999999999999992


In [38]:
result_df = pd.DataFrame(

    results,

    columns=[
        "ConvNeXt",
        "Swin",
        "SigLIP",
        "MacroF1"
    ]

)

result_df = result_df.sort_values(

    "MacroF1",

    ascending=False

)

display(result_df.head(20))

,ConvNeXt,Swin,SigLIP,MacroF1
178,0.55,0.35,0.10,0.959520
177,0.55,0.30,0.15,0.958987
186,0.60,0.30,0.10,0.958987
144,0.40,0.40,0.20,0.958461
166,0.50,0.30,0.20,0.958440
187,0.60,0.35,0.05,0.958435
191,0.65,0.15,0.20,0.958434
167,0.50,0.35,0.15,0.958416
155,0.45,0.35,0.20,0.957887
165,0.50,0.25,0.25,0.957887


In [39]:
best_probs = (

    best_weight[0] * p_conv_test +

    best_weight[1] * p_swin_test +

    best_weight[2] * p_siglip_test

)

predictions = best_probs.argmax(axis=1)

In [40]:
submission = pd.read_csv("../data/submission.csv")

submission["predicted"] = predictions

submission.to_csv(

    "submission_best_weight.csv",

    index=False

)

print("Saved submission_best_weight.csv")

Saved submission_best_weight.csv
